<a href="https://colab.research.google.com/github/ltz22/Neural-Network-Quantum-States/blob/main/3x3_Fresh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup


In [16]:
import jax
jax.config.update("jax_enable_x64", True)  # you already tested this
import jax.numpy as jnp
from jax import random
import itertools
import numpy as np

L = 4
N = L * L
J = 1.0
h = 0.5
EXACT_E0 = -18.281621

#generate pseudo-random key
key = random.PRNGKey(0)
#number of channels
C = 4
k1, k2 = random.split(key)
params = {
    'W': random.normal(k1, (C, 2, 2)) * 0.1,
    'b': random.normal(k2, (C,)) * 0.1, }


Convolution filter and energy calculation

In [17]:
def periodic_conv_features(spins_grid, W, b):
    C = W.shape[0]
    padded = jnp.pad(spins_grid, ((0, 1), (0, 1)), mode='wrap')
    feats = []
    for c in range(C):
        out = (W[c, 0, 0] * padded[0:L, 0:L]     + W[c, 0, 1] * padded[0:L, 1:L+1] +
               W[c, 1, 0] * padded[1:L+1, 0:L]    + W[c, 1, 1] * padded[1:L+1, 1:L+1] + b[c])
        feats.append(out)
    return jnp.stack(feats, axis=0)

def raw_network(params, spins_grid):
    feats = periodic_conv_features(spins_grid, params['W'], params['b'])
    return jnp.sum(jnp.log(jnp.cosh(feats)))

def log_psi(params, spins_grid):
    raw_pos = raw_network(params, spins_grid)
    raw_neg = raw_network(params, -spins_grid)
    return jax.scipy.special.logsumexp(jnp.array([raw_pos, raw_neg]))

def local_energy(params, spins_grid):
    e_diag = -J * jnp.sum(spins_grid * jnp.roll(spins_grid, -1, axis=1)) \
             -J * jnp.sum(spins_grid * jnp.roll(spins_grid, -1, axis=0))
    log_psi_s = log_psi(params, spins_grid)
    def flip_and_logpsi(r, c):
        flipped = spins_grid.at[r, c].multiply(-1)
        return log_psi(params, flipped)
    rs, cs = jnp.meshgrid(jnp.arange(L), jnp.arange(L), indexing='ij')
    log_psi_flips = jax.vmap(flip_and_logpsi)(rs.flatten(), cs.flatten())
    e_offdiag = -h * jnp.sum(jnp.exp(log_psi_flips - log_psi_s))
    return e_diag + e_offdiag

Made up optimisation

In [18]:
log_psi_jit = jax.jit(log_psi)
local_energy_jit = jax.jit(local_energy)
grad_log_psi_jit = jax.jit(jax.grad(log_psi))

log_psi_vmap = jax.vmap(log_psi_jit, in_axes=(None, 0))
local_energy_vmap = jax.vmap(local_energy_jit, in_axes=(None, 0))
grad_log_psi_vmap = jax.vmap(grad_log_psi_jit, in_axes=(None, 0))

MCMC

In [19]:
def metropolis_chain(params, key, init_spins, n_steps):
    log_psi_init = log_psi_jit(params, init_spins) #compute an initial log_psi with randomised initial spins

    #Call this step function inside the chain function n_steps times
    def step(carry, key):
        spins, lp = carry #carry is composed of (spins_config, log_psi) after each step
        k_site, k_accept = random.split(key) #create random keys on what to flip and to determine whether to accept
        site = random.randint(k_site, (), 0, N) #using k_site to generate a random site within the dimension L
        r, c = site // L, site % L
        proposal = spins.at[r, c].multiply(-1) #flip the site
        lp_prop = log_psi_jit(params, proposal) #Compute proposed log probability
        log_ratio = 2 * (lp_prop - lp) #Compute a ratio of proposed log prob and original log prob (2 * for squared)
        accept = jnp.log(random.uniform(k_accept)) < log_ratio #if the log of k_accept is less than ratio, accept (so that we have random acceptance)
        new_spins = jnp.where(accept, proposal, spins) #If accept, go with the proposed spins one, if not, stick with original spins
        new_lp = jnp.where(accept, lp_prop, lp) #Same thing as above
        return (new_spins, new_lp), new_spins #Return spins and log psi as carry (which goes to the next iteration), new_spins goes into a record of trajectory

    keys = random.split(key, n_steps) #Ensure every step has a different key
    (_, _), traj = jax.lax.scan(step, (init_spins, log_psi_init), keys) #Optimised for-loop to call step iteratively, append state after each iteration in traj
    return traj  # (n_steps, L, L)

Sampling

In [20]:

#create multiple chains of sample from metropolis_chain
#burn_in ignores initial steps to elimitate initialisation bias, n_keep is how many to keep, thin is spacing between kept samples
#This reduces correlation between samples
def sample_states(params, key, n_chains, burn_in, n_keep, thin):
    n_steps = burn_in + n_keep * thin #Compute how many steps to take
    k_init, k_chains = random.split(key)
    init_keys = random.split(k_init, n_chains) #Each chain gets a unique random initial key
    step_keys = random.split(k_chains, n_chains) #Each step gets a key as well

    init_spins = jax.vmap(lambda k: random.choice(k, jnp.array([-1, 1]), (L, L)))(init_keys) #For every chain, returns a randomised grid of spin configuration
    trajs = jax.vmap(metropolis_chain, in_axes=(None, 0, 0, None))(params, step_keys, init_spins, n_steps) #Run all metropolis_chain
    #Keep params and n_steps the same, change step_keys and give each chain a different starting lattice everytime
    # trajs: (n_chains, n_steps, L, L) -> discard burn-in, thin, flatten chains
    kept = trajs[:, burn_in::thin, :, :] #Only for the step attribute(second dimension), start and burn_in and thin
    return kept.reshape(-1, L, L) #Concatenate chains into many samples of spin lattices

SGD function

In [21]:
#set chaining parameters
def mc_expectation_step(params, key, n_chains=64, burn_in=50, n_keep=40, thin=5):
    samples = sample_states(params, key, n_chains, burn_in, n_keep, thin)
    n_total = samples.shape[0] #obtain all the samples and total number of samples

    E_locs = local_energy_vmap(params, samples) # compute local energy for every sample, results in a n_chains * n_keep rows vector
    grads = grad_log_psi_vmap(params, samples) #Calculate gradient for every sample (how does wavefunction change when changing the parameters)
    mean_E = jnp.mean(E_locs) #average all sample energies to produce final estimate
    min_E = jnp.min(E_locs)
    grad_update = {} #empty dictionary to store params update
    #Loop over weights first, then biases
    for kname in params:
        g = grads[kname] #get gradient for this parameter
        mean_grad = jnp.mean(g, axis=0) #Take the average gradient over all samples
        E_bcast = E_locs.reshape((n_total,) + (1,) * (g.ndim - 1)) #Do some dimension reshaping tricks to make multiplication work
        mean_E_grad = jnp.mean(E_bcast * g, axis=0)
        grad_update[kname] = 2 * (mean_E_grad - mean_E * mean_grad) #Compute final gradient update by some random formula

    return mean_E, grad_update, min_E

Iterations

In [22]:

#Learning rate, for now constant
lr = 0.01
n_iterations = 500
for it in range(n_iterations):
    k2, subkey = random.split(k2)
    mean_E, grad_update, min_E = mc_expectation_step(params, subkey)
    grad_norm = jnp.sqrt(sum(jnp.sum(g**2) for g in grad_update.values()))
    W_norm = jnp.sqrt(jnp.sum(params['W']**2))

    #gradient descent
    params = {k: params[k] - lr * grad_update[k] for k in params}

    if it % 10 == 0 or it == n_iterations - 1:
        print(f"iter {it:4d} E={mean_E:.10f} |grad|={grad_norm:.3e} min_E ={min_E:.10f}")

iter    0 E=-7.2707710066 |grad|=3.207e+01 min_E =-31.1698035074
iter    1 E=-22.1462494612 |grad|=1.041e+02 min_E =-34.5005984349
iter    2 E=-42.3519409215 |grad|=1.437e+02 min_E =-767.7609543134
iter    3 E=-31.7607470098 |grad|=2.025e+01 min_E =-32.9875527790
iter    4 E=-32.5248064235 |grad|=3.948e+00 min_E =-47.5635105086
iter    5 E=-32.5881986465 |grad|=3.197e+00 min_E =-40.1733020300
iter    6 E=-32.5093438322 |grad|=1.107e+00 min_E =-35.6141252729
iter    7 E=-32.4946448564 |grad|=1.190e-01 min_E =-34.2669469177
iter    8 E=-32.5099483878 |grad|=3.415e-01 min_E =-34.2004578554
iter    9 E=-32.5061230457 |grad|=4.202e-01 min_E =-33.8292559265
iter   10 E=-32.5041540220 |grad|=1.343e-01 min_E =-33.3458704768
iter   11 E=-32.5039277555 |grad|=1.472e-01 min_E =-33.2040465973
iter   12 E=-32.4817658358 |grad|=4.679e-01 min_E =-33.0769433248
iter   13 E=-32.5029100925 |grad|=1.367e-01 min_E =-33.5468316735
iter   14 E=-32.4805210744 |grad|=5.943e-01 min_E =-33.4242696750
iter   15 

KeyboardInterrupt: 

In [23]:
!git clone https://github.com/ltz22/Neural-Network-Quantum-States.git


Cloning into 'Neural-Network-Quantum-States'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 14 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 33.52 KiB | 591.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.


In [24]:
%cd Neural-Network-Quantum-States/


/content/Neural-Network-Quantum-States


In [27]:
!git add .

In [30]:
!git commit -m "Added 3 by 3 ising model with unique key and resampling every iteration"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [29]:
!git config --global user.email "zhangletong07@gmail.com"